In [12]:
import os
import numpy as np
import pandas as pd 
import matplotlib.pyplot as plt

import mlflow

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression,Lasso,Ridge
from sklearn.metrics import mean_squared_error, r2_score,mean_absolute_error
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor


In [13]:
df=pd.read_csv(r"D:\MY Projects (github)\mlops_day1\data\data.csv") 
df.head()

,TV,Radio,Newspaper,Sales
0,230.1,37.8,69.2,22.1
1,44.5,39.3,45.1,10.4
2,17.2,45.9,69.3,12.0
3,151.5,41.3,58.5,16.5
4,180.8,10.8,58.4,17.9


In [14]:
X=df[["TV","Radio","Newspaper"]]
y=df[["Sales"]]
Xtrain,Xtest,ytrain,ytest=train_test_split(X,y,test_size=0.2,random_state=42)

first,we configure mlflow to store all the experiment metadata inside a local SqLlite database named mlflow.db located in the current working directory.

In [15]:
mlflow.set_tracking_uri("sqlite:///mlflow.db")

Than we create an experiment named Advertising sales prediction.

In [16]:
mlflow.set_experiment("Advertising sales prediction")

<Experiment: artifact_location='file:d:/MY Projects (github)/mlops_day1/notebooks/mlruns/1', creation_time=1788408014149, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1788408014149, lifecycle_stage='active', name='Advertising sales prediction', tags={}, trace_location=None, workspace='default'>

Now we create a run inside the experiment Advertising sales prediction.

In [18]:
 with mlflow.start_run(run_name="Linear Regression"): 
    model=LinearRegression()
    model.fit(Xtrain,ytrain)
    ypred=model.predict(Xtest)
    
    rmse = np.sqrt(mean_squared_error(ytest, ypred))
    r2 = r2_score(ytest, ypred)
    
    print("RMSE:", rmse)
    print("R²:", r2)
    
    # Parameters
    mlflow.log_param("model", "Linear Regression")
    mlflow.log_param("test_size", 0.2)
    mlflow.log_param("random_state", 42)

    # Metrics:

    mlflow.log_metric("test_rmse", rmse)
    mlflow.log_metric("test_r2", r2)

RMSE: 1.705214622934923
R²: 0.9059011844150826


now we create another run using a ridge regression model.

In [19]:
with mlflow.start_run(run_name="Ridge Regression"):
    model=Ridge(alpha=1.0)
    model.fit(Xtrain,ytrain)
    ypred=model.predict(Xtest)
    
    rmse = np.sqrt(mean_squared_error(ytest, ypred))
    r2 = r2_score(ytest, ypred)
    
    print("RMSE:", rmse)
    print("R²:", r2)
    
    # Parameters
    mlflow.log_param("model", "Ridge Regression")
    mlflow.log_param("alpha", 1.0)
    mlflow.log_param("test_size", 0.2)
    mlflow.log_param("random_state", 42)

    # Metrics:

    mlflow.log_metric("test_rmse", rmse)
    mlflow.log_metric("test_r2", r2)

    #Log artifacts   skops is the model file.
    mlflow.sklearn.log_model(sk_model=model,name="ridge_model")

RMSE: 1.7052261161989772
R²: 0.9058999159458062


In [20]:
# turn on scikit-learn autologging.
mlflow.sklearn.autolog()

In [22]:
with mlflow.start_run(run_name="Random Forest Regression") as run:
    model=RandomForestRegressor(n_estimators=100, random_state=42)
    model.fit(Xtrain,ytrain)
    ypred=model.predict(Xtest)
    
    rmse = np.sqrt(mean_squared_error(ytest, ypred))
    r2 = r2_score(ytest, ypred)
    
    
    test_rsme=root_mean_squared_error(ytest, ypred)
    test_mae=mean_absolute_error(ytest, ypred)
    test_r2=r2_score(ytest, ypred)
    
    # custom project metrics
    mlflow.log_metrics({
        "test_rmse": test_rsme,
        "test_mae": test_mae,
        "test_r2": test_r2
    })

d:\MY Projects (github)\mlops_day1\workshop_mlops\Lib\site-packages\sklearn\base.py:1403: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)
2026/09/03 20:31:09 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


# Registering the model

the mlflow model is registry is a central hub(like an app store or catalog)  to keep all production ready models in one shared searchable palce istead of scattered across folders or runs.


among all the experiments you perform register the final selected model.



in a production environment, the models are continuously trained.This means the registered models would have many versions :



advertising sales_model
|
|----v1
|----v2 
|----v3
|----v4

In [24]:
# for registering the model , we require the model URI:
#URI is a unique identifier for the model.


run_id=run.info.run_id
model_uri=f"runs:/{run_id}/model"
print(model_uri)

runs:/da9488cc1cfe4c9f94901e79a389a61c/model


In [25]:
# now lets register the model.

registered_model=mlflow.register_model(model_uri=model_uri, name="Advertising_Sales_Model")
print(registered_model)

Successfully registered model 'Advertising_Sales_Model'.
2026/09/03 20:50:15 WARNING mlflow.tracking._model_registry.fluent: Run with id da9488cc1cfe4c9f94901e79a389a61c has no artifacts at artifact path 'model', registering model based on models:/m-d62203df4f8c4188a68783c5f0874bc5 instead


<ModelVersion: aliases=[], creation_timestamp=1788448815876, current_stage='None', deployment_job_state=None, description=None, last_updated_timestamp=1788448815876, metrics=None, model_id=None, name='Advertising_Sales_Model', params=None, run_id='da9488cc1cfe4c9f94901e79a389a61c', run_link=None, source='models:/m-d62203df4f8c4188a68783c5f0874bc5', status='READY', status_message=None, tags={}, user_id=None, version=1, workspace='default'>


Created version '1' of model 'Advertising_Sales_Model'.


# Model aliases

- model aliases gives a nickname(like current best or production) to a specific version of the registered model.

- Instead of typing exact numbers like version1,version 2 or version 15 you just use the nickname

- **champion** means: the currently preferred model
- **challenger** means: a new candidate being evaluated as a possible replacement.

In [30]:
from mlflow import MlflowClient
client=MlflowClient()

client.set_registered_model_alias(
    name="Advertising_Sales_Model",
    alias="champion",
    version="1"
)

Loading the registered model.

In [31]:
model=mlflow.sklearn.load_model(
    "models:/Advertising_Sales_Model@champion"
)

# generate new prediction

new_data=pd.DataFrame({
    "TV":[150,0],
    "Radio":[30,0],
    "Newspaper":[10,0]
})

print(model.predict(new_data))

[15.629  3.2  ]
